In [ ]:
# ============================================================
# Spectrometer ray-trace (2D paraxial)
# SLIT -> COLLIMATING LENS -> TRANSMISSION GRATING -> CAMERA LENS -> SENSOR
#
# Simplified version:
# 1) Uses only a 1 mm slit width
# 2) No slit sweep
# 3) No graphs
# 4) Prints only geometry + performance outputs
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
# ============================================================
# SETTINGS
# ============================================================
MODE = "final"   # "optimize" or "final"

SLIT_WIDTH_UM = 1000.0   # 1 mm slit
sample_state = "70h"

# ============================================================
# iPhone 12 mini sensor parameters
# ============================================================
sensor_width = 0.00581   # m
pixel_size   = 1.4e-6    # m
sensor_half  = sensor_width / 2

# ============================================================
# Optics
# ============================================================
# Collimator lens (AC050-008-A)
f_coll = 7.5e-3
D_coll = 5e-3
R_coll = D_coll / 2

# Camera lens (iPhone 12 mini)
f_cam = 5.0e-3
f_number = 1.6
D_cam = f_cam / f_number
R_cam = D_cam / 2

# Transmission grating (1200 lines/mm)
lines_per_mm = 1200
d_gr = 1.0 / (lines_per_mm * 1e3)
m_order = 1

# Slit plane
x_slit = 0.0

# ============================================================
# RGB weighting proxy
# ============================================================
RGB_means = {
    "2h":  (142.046432, 136.226868, 126.367401),
    "24h": (123.136177, 105.135864,  93.817390),
    "70h": (100.529762,  70.482079,  62.534435),
}

def rgb_to_weight_spectrum(mean_R, mean_G, mean_B, lam_nm):
    r = mean_R / 255.0
    g = mean_G / 255.0
    b = mean_B / 255.0

    anchors_lam = np.array([460.0, 550.0, 650.0])
    anchors_w   = np.array([b, g, r])

    w = np.interp(lam_nm, anchors_lam, anchors_w)
    w = np.clip(w, 0.0, 1.0)
    if w.max() > 0:
        w = w / w.max()
    return w

# ============================================================
# MODE SETTINGS
# ============================================================
if MODE == "optimize":
    lam = np.linspace(420e-9, 680e-9, 81)
    scatter_half_angle_deg = 18.0
    N_angle = 41
    N_ap = 21

    L_cg_coarse = np.linspace(2e-3, 12e-3, 11)
    L_gc_coarse = np.linspace(2e-3, 12e-3, 11)
    refine_halfspan_mm = 1.0
    refine_steps = 9
    WEIGHT_SKIP = 0.05

elif MODE == "final":
    lam = np.linspace(400e-9, 700e-9, 201)
    scatter_half_angle_deg = 20.0
    N_angle = 81
    N_ap = 41

    L_cg_coarse = np.linspace(2e-3, 12e-3, 21)
    L_gc_coarse = np.linspace(2e-3, 12e-3, 21)
    refine_halfspan_mm = 0.5
    refine_steps = 11
    WEIGHT_SKIP = 0.00
else:
    raise ValueError("MODE must be 'optimize' or 'final'")

lam_nm  = lam * 1e9
lam0_nm = 550.0
lam0    = lam0_nm * 1e-9
mid_idx = np.argmin(np.abs(lam_nm - lam0_nm))

w_spec = rgb_to_weight_spectrum(*RGB_means[sample_state], lam_nm)

# ============================================================
# Functions
# ============================================================
def normalize(v):
    v = np.array(v, dtype=float)
    n = np.linalg.norm(v)
    return v if n == 0 else v / n

def intersect_xplane(p, d, x_plane):
    if abs(d[0]) < 1e-12:
        return None, None
    t = (x_plane - p[0]) / d[0]
    return p + t * d, t

def thin_lens_outdir(p_hit, d_in, f):
    """
    Paraxial thin-lens model:
        theta_out = theta_in - y/f
    """
    y = p_hit[1]
    theta_in = np.arctan2(d_in[1], d_in[0])
    theta_out = theta_in - y / f
    return normalize([np.cos(theta_out), np.sin(theta_out)])

def rotate_dir(d, psi):
    c, s = np.cos(psi), np.sin(psi)
    return normalize([c*d[0] - s*d[1], s*d[0] + c*d[1]])

def grating_theta_m_transmission(theta_i, lam_m, d, m=1):
    rhs = np.sin(theta_i) + (m * lam_m / d)
    if abs(rhs) > 1.0:
        return None
    return np.arcsin(rhs)

def solve_theta_i_for_centering_transmission(lam_target, d, m, theta_m_target=0.0):
    val = np.sin(theta_m_target) - (m * lam_target / d)
    if abs(val) > 1.0:
        return None
    return np.arcsin(val)

# ============================================================
# Fixed slit width
# ============================================================
slit_w = SLIT_WIDTH_UM * 1e-6

def generate_slit_rays():
    """
    Adaptive slit sampling:
    wider slits get more sampled points.
    """
    n_slit_pts = max(11, int(np.ceil(slit_w / 50e-6)) + 1)
    ys = np.linspace(-slit_w/2, +slit_w/2, n_slit_pts)

    half = np.deg2rad(scatter_half_angle_deg)
    angles = np.linspace(-half, +half, N_angle)

    rays = []
    for y in ys:
        p = np.array([x_slit, y], float)
        for a in angles:
            rays.append((p, normalize([np.cos(a), np.sin(a)])))
    return rays

# ============================================================
# Ray tracing core
# ============================================================
def trace_one_wavelength(L_cg, L_gc, theta_i0, lam_m, entrance_rays):
    x_coll   = x_slit + f_coll
    x_gr     = x_coll + L_cg
    x_cam    = x_gr   + L_gc
    x_sensor = x_cam  + f_cam

    y_ap = np.linspace(-R_coll, +R_coll, N_ap)

    hits = []

    d1_center = thin_lens_outdir(np.array([x_coll, 0.0]), np.array([1.0, 0.0]), f_coll)
    theta_center = np.arctan2(d1_center[1], d1_center[0])
    psi = theta_i0 - theta_center

    for (p_slit, _) in entrance_rays:
        for ya in y_ap:
            p_target = np.array([x_coll, ya], float)
            d0 = normalize(p_target - p_slit)

            p_coll, t = intersect_xplane(p_slit, d0, x_coll)
            if p_coll is None or t <= 0 or abs(p_coll[1]) > R_coll:
                continue

            d1 = thin_lens_outdir(p_coll, d0, f_coll)
            d1 = rotate_dir(d1, psi)

            p_gr, t = intersect_xplane(p_coll, d1, x_gr)
            if p_gr is None or t <= 0:
                continue

            theta_i = np.arctan2(d1[1], d1[0])
            theta_m = grating_theta_m_transmission(theta_i, lam_m, d_gr, m=m_order)
            if theta_m is None:
                continue
            d2 = normalize([np.cos(theta_m), np.sin(theta_m)])

            p_cam, t = intersect_xplane(p_gr, d2, x_cam)
            if p_cam is None or t <= 0 or abs(p_cam[1]) > R_cam:
                continue

            d3 = thin_lens_outdir(p_cam, d2, f_cam)

            p_s, t = intersect_xplane(p_cam, d3, x_sensor)
            if p_s is None or t <= 0:
                continue

            y_on_sensor = p_s[1]
            if abs(y_on_sensor) <= sensor_half:
                hits.append(y_on_sensor)

    return hits

def trace_system(L_cg, L_gc, theta_i0, entrance_rays, w_spec):
    centroid = np.full_like(lam, np.nan, dtype=float)
    rms      = np.full_like(lam, np.nan, dtype=float)
    counts   = np.zeros_like(lam, dtype=int)
    signal_w = np.zeros_like(lam, dtype=float)

    for i, L in enumerate(lam):
        wi = float(w_spec[i])
        if wi < WEIGHT_SKIP:
            continue

        hits = trace_one_wavelength(L_cg, L_gc, theta_i0, L, entrance_rays)
        if hits:
            hits = np.array(hits)
            centroid[i] = hits.mean()
            rms[i] = hits.std()
            counts[i] = len(hits)
            signal_w[i] = wi * len(hits)

    return centroid, rms, counts, signal_w

# ============================================================
# Resolution estimation
# ============================================================
def local_dispersion_at_550(centroid_m):
    ok = np.isfinite(centroid_m)
    if ok.sum() < 10:
        return np.nan, np.nan

    c = centroid_m.copy()
    c[~ok] = np.interp(lam_nm[~ok], lam_nm[ok], c[ok])

    dxdnm = np.gradient(c, lam_nm)   # m / nm
    dxdnm_mid = dxdnm[mid_idx]

    if not np.isfinite(dxdnm_mid) or abs(dxdnm_mid) < 1e-15:
        return np.nan, np.nan

    nm_per_pixel = pixel_size / abs(dxdnm_mid)
    return nm_per_pixel, dxdnm_mid

def estimate_resolution_metrics(centroid, rms, signal_w):
    coverage = np.isfinite(centroid).mean()
    throughput_w = np.nanmedian(signal_w[np.isfinite(signal_w)]) if np.isfinite(signal_w).any() else 0.0

    nm_pix, dxdnm_mid = local_dispersion_at_550(centroid)
    blur_mid = rms[mid_idx] if np.isfinite(rms[mid_idx]) else np.nan

    # Simplified slit image assumption
    slit_image = slit_w

    if np.isfinite(blur_mid) and np.isfinite(dxdnm_mid) and abs(dxdnm_mid) > 0:
        spot = np.sqrt(slit_image**2 + blur_mid**2)
        dlam_nm = spot / abs(dxdnm_mid)
        R550 = lam0_nm / dlam_nm if dlam_nm > 0 else np.nan
    else:
        dlam_nm = np.nan
        R550 = np.nan

    return coverage, throughput_w, nm_pix, dlam_nm, R550

def score_geometry(centroid, rms, signal_w):
    coverage, throughput_w, nm_pix, dlam_nm, R550 = estimate_resolution_metrics(centroid, rms, signal_w)

    score = (
        2.5 * coverage
        + 1.0 * np.log10(throughput_w + 1.0)
        - (0.15 * np.log10(dlam_nm + 1.0) if np.isfinite(dlam_nm) else 10.0)
    )

    return score, coverage, throughput_w, nm_pix, dlam_nm, R550

# ============================================================
# Choosing grating incidence so 550 nm is near centre
# ============================================================
theta_i0 = solve_theta_i_for_centering_transmission(lam0, d_gr, m_order, theta_m_target=0.0)
if theta_i0 is None:
    theta_i0 = np.deg2rad(-20.0)

# ============================================================
# GEOMETRY SEARCH
# ============================================================
entrance_rays = generate_slit_rays()

print(f"[{MODE}] Slit width = {SLIT_WIDTH_UM:.0f} µm")
print(f"[{MODE}] Rays launched = {len(entrance_rays)}")
print(f"[{MODE}] Sample weighting state = {sample_state}")
print(f"[{MODE}] Camera f/# = f/{f_number:.1f}")

best = None
for L_cg in L_cg_coarse:
    for L_gc in L_gc_coarse:
        total_len = f_coll + L_cg + L_gc + f_cam
        if not (20e-3 <= total_len <= 40e-3):
            continue

        centroid, rms, counts, signal_w = trace_system(L_cg, L_gc, theta_i0, entrance_rays, w_spec)
        score, coverage, throughput_w, nm_pix, dlam_nm, R550 = score_geometry(centroid, rms, signal_w)

        cand = (score, L_cg, L_gc, total_len, coverage, throughput_w, nm_pix, dlam_nm, R550)
        if (best is None) or (cand[0] > best[0]):
            best = cand

score1, L_cg1, L_gc1, total_len1, cov1, thr1, nm_pix1, dlam1, R1 = best
print(f"\n[{MODE}] Coarse best: L_cg = {L_cg1*1e3:.2f} mm, L_gc = {L_gc1*1e3:.2f} mm")

span = refine_halfspan_mm * 1e-3
L_cg_ref = np.linspace(max(0.5e-3, L_cg1 - span), L_cg1 + span, refine_steps)
L_gc_ref = np.linspace(max(0.5e-3, L_gc1 - span), L_gc1 + span, refine_steps)

best2 = None
for L_cg in L_cg_ref:
    for L_gc in L_gc_ref:
        total_len = f_coll + L_cg + L_gc + f_cam
        if not (20e-3 <= total_len <= 40e-3):
            continue

        centroid, rms, counts, signal_w = trace_system(L_cg, L_gc, theta_i0, entrance_rays, w_spec)
        score, coverage, throughput_w, nm_pix, dlam_nm, R550 = score_geometry(centroid, rms, signal_w)

        cand = (score, L_cg, L_gc, total_len, coverage, throughput_w, nm_pix, dlam_nm, R550)
        if (best2 is None) or (cand[0] > best2[0]):
            best2 = cand

score, L_cg_best, L_gc_best, total_len, coverage, throughput_w, nm_pix, dlam_nm, R550 = best2

# ============================================================
# FINAL OUTPUT
# ============================================================
print("\n==============================")
print(f"Spectrometer ({MODE.upper()}) — BEST GEOMETRY")
print("==============================")
print(f"Slit width: {SLIT_WIDTH_UM:.0f} µm")
print(f"Fan half-angle: {scatter_half_angle_deg:.1f} deg")
print(f"Nominal θ_i0: {np.rad2deg(theta_i0):.2f} deg")
print("")
print("Apertures:")
print(f"  Collimator: Ø{D_coll*1e3:.2f} mm (R={R_coll*1e3:.2f} mm)")
print(f"  Camera:     f/{f_number:.1f} -> D={D_cam*1e3:.3f} mm (R={R_cam*1e3:.3f} mm)")
print("")
print("Best geometry:")
print(f"  slit -> collimator:     {f_coll*1e3:.2f} mm (fixed)")
print(f"  collimator -> grating:  {L_cg_best*1e3:.2f} mm")
print(f"  grating -> camera lens: {L_gc_best*1e3:.2f} mm")
print(f"  camera -> sensor:       {f_cam*1e3:.2f} mm (fixed)")
print(f"  total slit->sensor len: {total_len*1e3:.2f} mm")
print("")
print(f"Coverage: {coverage:.2f}")
print(f"Weighted throughput proxy (median): {throughput_w:.2f}")
print(f"Midband nm/pixel: {nm_pix:.3f} nm/px")
print(f"Estimated Δλ at 550 nm: {dlam_nm:.3f} nm")
print(f"Estimated resolving power at 550 nm: R ~ {R550:.0f}")
print("==============================")


[optimize] Slit width = 1000 µm
[optimize] Rays launched = 861
[optimize] Sample weighting state = 70h
[optimize] Camera f/# = f/1.6


KeyboardInterrupt: 